In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Transforma o formato do DF de "largo" para "longo" (melt)
# Isso junta todas as colunas de avaliação em uma só para o Seaborn ler
df_longo = df.melt(
    value_vars=["review_score", "sentimento_vader", "sentimento_nota_leia", "sentimento_nota_textblob", "sentimento_nota_transformer"],
    var_name="Origem_da_Nota",
    value_name="Nota_Atribuida",
)

# 2. Configura o estilo visual do gráfico
sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 7))

# 3. Plota o gráfico de colunas agrupadas
# Mudamos a paleta para destacar bem a 'review_score' dos modelos
ax = sns.countplot(
    data=df_longo, x="Nota_Atribuida", hue="Origem_da_Nota", palette="Set2"
)

# 4. Ajustes estéticos e títulos
plt.title(
    "Comparação de Distribuição: Notas Originais vs. Modelos de IA",
    fontsize=16,
    weight="bold",
    pad=20,
)
plt.xlabel("Notas de Avaliação de Sentimento (1 a 5)", fontsize=13, labelpad=10)
plt.ylabel("Quantidade de Comentários", fontsize=13, labelpad=10)

# Melhora o posicionamento e título da legenda
plt.legend(title="Origem do Resultado", fontsize=11, title_fontsize=12)

# Adiciona os valores exatos em cima de cada barra para facilitar a leitura
for p in ax.patches:
    if p.get_height() > 0:  # Evita erro se a altura for zero
        ax.annotate(
            f"{int(p.get_height())}",
            (p.get_x() + p.get_width() / 2.0, p.get_height()),
            ha="center",
            va="center",
            xytext=(0, 5),
            textcoords="offset points",
            fontsize=9,
        )

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# Cria tabela de notas totalmente opostas entre nota original e modelos
# O foco é contar apenas os casos extremos 1 vs 5 para cada nota original

def contar_notas_opostas(df, modelo):
    mask = ((df['review_score'] == 1) & (df[modelo] == 5)) | ((df['review_score'] == 5) & (df[modelo] == 1))
    return df[mask].groupby('review_score').size().reindex([1, 2, 3, 4, 5], fill_value=0)

# Calcula a tabela de totais opostos
modelos = ['sentimento_vader', 'sentimento_nota_leia', 'sentimento_nota_textblob', 'sentimento_nota_transformer']
df_opostos = pd.DataFrame({
    modelo: contar_notas_opostas(df, modelo) for modelo in modelos
})
df_opostos.index.name = 'review_score'
df_opostos = df_opostos.loc[[1, 5]].reset_index()

print('Total de notas totalmente opostas (1 vs 5) por nota original e modelo:')
display(df_opostos)


In [ ]:
import pandas as pd

# Cria tabela de notas totalmente opostas entre nota original e modelos
def contar_notas_opostas(df, modelo):
    mask = ((df['review_score'] == 1) & (df[modelo] == 5)) | ((df['review_score'] == 5) & (df[modelo] == 1))
    return df[mask].groupby('review_score').size().reindex([1, 2, 3, 4, 5], fill_value=0)

df_opostos = pd.DataFrame({
    modelo: contar_notas_opostas(df, modelo) for modelo in ['sentimento_vader', 'sentimento_nota_leia', 'sentimento_nota_textblob', 'sentimento_nota_transformer']
})
df_opostos.index.name = 'review_score'
df_opostos = df_opostos.loc[[1, 5]].reset_index()

print('Total de notas totalmente opostas (1 vs 5) por nota original e modelo:')
display(df_opostos)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Prepara os dados para o gráfico de linhas (calcula a média de acertos por faixa)
colunas_acerto = ["Acertou_sentimento_vader", "Acertou_sentimento_nota_leia", "Acertou_sentimento_nota_textblob", "Acertou_sentimento_nota_transformer"]
df_performance = df_analise.groupby("Faixa_Tamanho", observed=False)[colunas_acerto].mean() * 100
df_performance = df_performance.reset_index().melt(id_vars="Faixa_Tamanho", var_name="Modelo", value_name="Acuracia")

# Renomeia os modelos para o gráfico ficar limpo
df_performance["Modelo"] = df_performance["Modelo"].str.replace("Acertou_", "")

# Plotando o gráfico
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

sns.lineplot(
    data=df_performance, 
    x="Faixa_Tamanho", 
    y="Acuracia", 
    hue="Modelo", 
    marker="o", # Adiciona uma bolinha em cada ponto
    linewidth=2.5
)

plt.title("Performance dos modelos de Acordo com o Tamanho do Comentário", fontsize=14, weight="bold", pad=15)
plt.xlabel("Tamanho do Comentário (Faixas de Palavras)", fontsize=12)
plt.ylabel("Taxa de Acerto (Acurácia em %)", fontsize=12)
plt.ylim(0, 100) # Mantém o eixo Y de 0 a 100%
plt.legend(title="Modelos")

plt.show()
